# Experiment 3 -- External validation on MIAS

**Question:** do the four deployed hi-res models still work on mammograms from a *different*
dataset than the one they were trained on?

Every number reported so far comes from the CBIS-DDSM test split -- same scanner pool, same
annotators, same preprocessing as training. That measures *internal* validity only. This notebook
evaluates the exact deployed checkpoints, **frozen and zero-shot** (no fine-tuning, no
recalibration), on the **MIAS** database (Mammographic Image Analysis Society, UK).

| | CBIS-DDSM (training) | MIAS (external) |
|---|---|---|
| Origin | USA (DDSM, 1990s) | UK national screening programme |
| Digitiser | Lumisys / Howtek / DBA scanners | Joyce-Loebl microdensitometer, 50 micron |
| Annotation | Lesion outlines (curated 2017) | Centre + radius circles |
| Labels | Pathology-verified | Radiologist truth data |
| Licence | TCIA | CC BY 2.0 UK |

**Protocol, fixed before looking at any MIAS result:**
- **Population:** every MIAS image that contains an abnormality. Normal images are excluded -- the
  models were trained on images that all contain a lesion, so "benign vs malignant given a
  finding" is the task they were built for.
- **Label:** malignant if any lesion on the image is malignant, otherwise benign.
- **Thresholds:** exactly as deployed (DenseNet121 0.305 from its checkpoint metadata, the others
  0.50), plus 0.50 for every model so all four are compared like for like.
- **Uncertainty:** 95% bootstrap CIs resampled **by woman** (MIAS IDs come in left/right pairs),
  not by image, because two breasts of one woman are not independent observations.
- **Localization:** Grad-CAM (malignant class, AMP off) scored against the MIAS lesion circles with
  the same `localization_metrics` used on CBIS-DDSM.

Citation: Suckling J. et al. (1994). *The Mammographic Image Analysis Society Digital Mammogram
Database.* Excerpta Medica, International Congress Series 1069, pp. 375-378.

In [ ]:
import os, sys, re, zipfile, warnings


def _find_repo_root(markers=("models", "dataframes", "notebooks")):
    """Walk up from the CWD -- the Jupyter kernel does not always start in this folder."""
    d = os.path.abspath(os.getcwd())
    for _ in range(6):
        if all(os.path.isdir(os.path.join(d, m)) for m in markers):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    raise RuntimeError(f"could not locate the repo root from {os.getcwd()!r}")


ROOT = _find_repo_root()
sys.path.insert(0, os.path.join(ROOT, "notebooks", "common"))

import numpy as np
import pandas as pd
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pypdf import PdfReader
from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import hires_lib as H

warnings.filterwarnings("ignore", category=FutureWarning)
H.set_seed()

MIAS_URL = ("https://api.repository.cam.ac.uk/server/api/core/bitstreams/"
            "5960ab2b-5ea2-4db1-96ac-15b3605e7485/content")
MIAS_DIR = os.path.join(ROOT, "MIAS")
MIAS_ZIP = os.path.join(MIAS_DIR, "miasdbv1.21.zip")
MIAS_IMG_DIR = os.path.join(MIAS_DIR, "MIASDBv1.21")
CACHE_DIR = os.path.join(ROOT, "mias-cache-hires")
TRUTH_CSV = os.path.join(ROOT, "data_description", "mias_truth_v1.21.csv")
MANIFEST_CSV = os.path.join(H.DF_DIR, "mias_df_hires.csv")
RESULTS_DIR = os.path.join(ROOT, "notebooks", "experiment_3_external_validation", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("root  :", ROOT)
print("device:", H.DEVICE)

## 1. Download MIAS

The Cambridge University repository hosts MIAS v1.21 (the original 50-micron images, not the
1024x1024 mini-MIAS) with an anonymous direct download. ~1.5 GB, skipped if already present.

In [ ]:
def download(url, dest, chunk=1 << 20):
    """Streamed download. A partial file from an interrupted run is detected by size and redone."""
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        if os.path.exists(dest) and total and os.path.getsize(dest) == total:
            print(f"already downloaded: {dest}")
            return
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest + ".part", "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                                    desc="MIAS") as bar:
            for b in r.iter_content(chunk):
                f.write(b)
                bar.update(len(b))
    os.replace(dest + ".part", dest)


if not os.path.isdir(MIAS_IMG_DIR):
    if not (os.path.exists(MIAS_ZIP) and zipfile.is_zipfile(MIAS_ZIP)):
        download(MIAS_URL, MIAS_ZIP)
    with zipfile.ZipFile(MIAS_ZIP) as z:
        z.extractall(MIAS_DIR)

PGMS = sorted(f for f in os.listdir(MIAS_IMG_DIR) if f.endswith(".pgm"))
assert len(PGMS) == 322, f"expected 322 MIAS images, found {len(PGMS)}"
print(f"{len(PGMS)} PGM images in {MIAS_IMG_DIR}")

## 2. Truth table

MIAS v1.21 ships its ground truth only as a table inside `00README.pdf`. Each line is
`ID tissue class severity x y radius`; an image with several lesions continues on lines that omit
the ID. Calcification clusters spread across the whole breast have no coordinates (6 images).
The parsed table is written to `data_description/` so the labels can be audited by hand.

In [ ]:
CLASSES = "CALC|CIRC|SPIC|MISC|ARCH|ASYM|NORM"
FIRST = re.compile(rf"^\s*(mdb\d{{3}}[lr][smlx])\s+([FGD])\s+({CLASSES})\b\s*([BM])?"
                   r"(?:\s+(\d+)\s+(\d+)\s+(\d+))?")
# Continuation lines carry a severity -- this is what separates them from the legend at the top of
# the README ("CALC - Calcification"), which would otherwise parse as a lesion.
CONT = re.compile(rf"^\s*({CLASSES})\s+([BM])(?:\s+(\d+)\s+(\d+)\s+(\d+))?")

text = "\n".join(p.extract_text() for p in PdfReader(os.path.join(MIAS_IMG_DIR, "00README.pdf")).pages)
rows, cur = [], None
for line in text.splitlines():
    if m := FIRST.match(line):
        cur = m.group(1), m.group(2)
        rows.append((*cur, m.group(3), m.group(4), *m.groups()[4:]))
    elif cur and (m := CONT.match(line)):
        rows.append((*cur, *m.groups()))

truth = pd.DataFrame(rows, columns=["id", "tissue", "abnorm_class", "severity", "x", "y", "radius"])
for c in ("x", "y", "radius"):
    truth[c] = pd.to_numeric(truth[c])
truth.to_csv(TRUTH_CSV, index=False)

assert truth["id"].nunique() == 322 and set(truth["id"] + ".pgm") == set(PGMS)
assert (truth["abnorm_class"] == "NORM").sum() == 207          # normal images
assert (truth["abnorm_class"] != "NORM").sum() == 123          # lesions (69 B + 54 M)
assert truth["severity"].value_counts().to_dict() == {"B": 69, "M": 54}
print(f"wrote {TRUTH_CSV}: {len(truth)} rows")
truth.groupby(["abnorm_class", "severity"], dropna=False).size().unstack(fill_value=0)

In [ ]:
les = truth[truth["abnorm_class"] != "NORM"]
img = (les.groupby("id")
          .agg(label=("severity", lambda s: int((s == "M").any())),
               abnorm_class=("abnorm_class", lambda s: "+".join(sorted(set(s)))),
               n_lesions=("severity", "size"),
               has_roi=("x", lambda s: bool(s.notna().all())))
          .reset_index())
# Odd/even IDs are the left/right films of one woman -- the bootstrap resamples these clusters.
img["woman_id"] = (img["id"].str[3:6].astype(int) + 1) // 2

print(f"abnormal images: {len(img)}  (benign {int((img.label == 0).sum())}, "
      f"malignant {int(img.label.sum())})  from {img.woman_id.nunique()} women")
print(f"images with lesion coordinates (localization subset): {int(img.has_roi.sum())}")
assert len(img) == 115 and img.label.sum() == 52

## 3. Coordinate convention and orientation

The mini-MIAS documentation puts the origin at the **bottom-left** corner; the v1.21 README does not
say. Getting this wrong flips every circle vertically and silently ruins the localization result,
so it is measured rather than assumed: for each lesion, compare the mean grey level inside the
circle under both conventions. A lesion sits inside breast tissue, so the correct convention puts
the circle on bright pixels, while the wrong one regularly lands it on the black film background.

In [ ]:
def load_pgm(name):
    with Image.open(os.path.join(MIAS_IMG_DIR, name + ".pgm")) as im:
        return im.convert("L")


sizes = {i: load_pgm(i).size for i in tqdm(img["id"], desc="sizes")}
img["src_w"] = img["id"].map(lambda i: sizes[i][0])
img["src_h"] = img["id"].map(lambda i: sizes[i][1])
print("source sizes (W x H):", sorted({s for s in sizes.values()}))


def circle_mean(arr, x, y, r):
    hh, ww = arr.shape
    yy, xx = np.ogrid[:hh, :ww]
    m = (xx - x) ** 2 + (yy - y) ** 2 <= r ** 2
    return float(arr[m].mean()) if m.any() else float("nan")


checks = []
for _, l in les.dropna(subset=["x"]).iterrows():
    a = np.asarray(load_pgm(l["id"]))
    hh = a.shape[0]
    checks.append({"id": l["id"],
                   "top": circle_mean(a, l.x, l.y, l.radius),
                   "bottom": circle_mean(a, l.x, hh - 1 - l.y, l.radius)})
checks = pd.DataFrame(checks)
bottom_wins = float((checks["bottom"] > checks["top"]).mean())
print(f"lesions where bottom-left origin gives the brighter circle: {bottom_wins:.1%}")
print(checks[["top", "bottom"]].describe().loc[["mean", "50%"]].round(1))

assert max(bottom_wins, 1 - bottom_wins) >= 0.75, "no clear winner -- inspect the overlays by hand"
Y_FROM_BOTTOM = bottom_wins > 0.5
print("convention:", "origin bottom-left (y flipped)" if Y_FROM_BOTTOM else "origin top-left")

In [ ]:
def lesion_mask(row):
    """Filled union of the image's lesion circles, drawn in SOURCE pixel space so the mask goes
    through exactly the same fit_pad as the image."""
    w, h = row["src_w"], row["src_h"]
    m = Image.new("L", (w, h), 0)
    d = ImageDraw.Draw(m)
    for _, l in les[(les["id"] == row["id"]) & les["x"].notna()].iterrows():
        cy = (h - 1 - l.y) if Y_FROM_BOTTOM else l.y
        d.ellipse([l.x - l.radius, cy - l.radius, l.x + l.radius, cy + l.radius], fill=255)
    return m


def outline(mask_arr):
    m = mask_arr > 127
    edge = m & ~(np.roll(m, 1, 0) & np.roll(m, -1, 0) & np.roll(m, 1, 1) & np.roll(m, -1, 1))
    return edge


sample = img[img.has_roi].sample(8, random_state=H.SEED)
fig, axes = plt.subplots(1, 8, figsize=(20, 5))
for ax, (_, r) in zip(axes, sample.iterrows()):
    small = (384, round(384 * r.src_h / r.src_w))
    a = np.asarray(load_pgm(r["id"]).resize(small, Image.BILINEAR))
    e = outline(np.asarray(lesion_mask(r).resize(small, Image.NEAREST)))
    rgb = np.stack([a] * 3, -1)
    rgb[e] = (57, 219, 92)
    ax.imshow(rgb)
    ax.set_title(f"{r['id']}\n{r.abnorm_class} {'M' if r.label else 'B'}", fontsize=9)
    ax.axis("off")
plt.suptitle("MIAS lesion circles in source space (check they sit on the lesion)")
plt.tight_layout()
plt.show()

### Orientation

MIAS stores every film **landscape** (4320 px wide), with the chest wall along the top edge. The
CBIS-DDSM training images are **portrait**, with the chest wall on one side and the pectoral muscle
in the upper corner. Fed as-is, `fit_pad` would shrink each MIAS breast into a thin horizontal
strip across the middle of the 384x640 canvas, which no model trained on CBIS-DDSM has ever seen.

Each image is therefore rotated 90 degrees counter-clockwise (chest wall moves to the left). Left
and right breasts then disagree on whether the pectoral corner is at the top or the bottom, so the
image is flipped vertically whenever the bottom-left corner is brighter than the top-left corner.
The lesion mask is drawn in source space and goes through exactly the same two operations, so it
stays pixel-aligned with the image.

In [ ]:
def chest_wall_on_top(a):
    return a[: a.shape[0] // 2].mean() > a[a.shape[0] // 2:].mean()


def needs_vflip(pil_src):
    """After the CCW rotation, is the bright pectoral corner at the bottom-left?"""
    r = np.asarray(pil_src.rotate(90, expand=True), dtype=np.float32)
    hh, ww = r.shape
    return bool(r[-int(0.3 * hh):, : int(0.25 * ww)].mean() > r[: int(0.3 * hh), : int(0.25 * ww)].mean())


def to_portrait(pil, vflip):
    out = pil.rotate(90, expand=True)          # CCW; exact for 90 degrees, no interpolation
    return out.transpose(Image.FLIP_TOP_BOTTOM) if vflip else out


flags = {}
for i in tqdm(img["id"], desc="orientation"):
    im = load_pgm(i)
    small = im.resize((im.size[0] // 8, im.size[1] // 8), Image.BILINEAR)
    flags[i] = (chest_wall_on_top(np.asarray(small, dtype=np.float32)), needs_vflip(small))
img["chest_wall_top"] = img["id"].map(lambda i: flags[i][0])
img["vflip"] = img["id"].map(lambda i: flags[i][1])
# After rotation, width and height swap. orig_w/orig_h are the PORTRAIT dimensions -- the image the
# model actually sees, and what H.validity_mask needs.
img["orig_w"], img["orig_h"] = img["src_h"], img["src_w"]

assert (img.src_w > img.src_h).all(), "expected every MIAS film to be stored landscape"
print(f"chest wall along the top edge: {img.chest_wall_top.mean():.1%} "
      f"(exceptions: {list(img.loc[~img.chest_wall_top, 'id'])})")
print(f"vertical flip applied: {int(img.vflip.sum())}/{len(img)}")

## 4. Preprocess to the training geometry

After the portrait rotation, identical to `build_hires_cache.py`: scale to fit 384x640 preserving aspect, centre on black --
LANCZOS for the image, NEAREST for the mask so it stays binary. The manifest uses the same column
names as the CBIS-DDSM hi-res dataframes, so `hires_lib.HiResDataset` loads it unchanged.
Resumable: existing cache files are kept.

In [ ]:
os.makedirs(os.path.join(CACHE_DIR, "full"), exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, "roi"), exist_ok=True)

full_paths, roi_paths = [], []
for _, r in tqdm(img.iterrows(), total=len(img), desc="cache"):
    fo = os.path.join(CACHE_DIR, "full", f"{r['id']}.png")
    ro = os.path.join(CACHE_DIR, "roi", f"{r['id']}.png")
    if not (os.path.exists(fo) and os.path.exists(ro)):
        H.fit_pad(to_portrait(load_pgm(r["id"]), r.vflip), Image.LANCZOS).save(fo, optimize=True)
        H.fit_pad(to_portrait(lesion_mask(r), r.vflip), Image.NEAREST).save(ro, optimize=True)
    full_paths.append(fo)
    roi_paths.append(ro)

mias_df = img.copy()
mias_df[H.FULL_HIRES_COL] = full_paths
mias_df[H.ROI_HIRES_COL] = roi_paths
mias_df[H.LABEL_COL] = mias_df["label"]
mias_df.to_csv(MANIFEST_CSV, index=False)
print(f"wrote {MANIFEST_CSV}: {len(mias_df)} rows")

for _, r in mias_df.iterrows():
    with Image.open(r[H.FULL_HIRES_COL]) as a, Image.open(r[H.ROI_HIRES_COL]) as m:
        assert a.size == (H.HIRES_W, H.HIRES_H) and m.size == (H.HIRES_W, H.HIRES_H)
        u = np.unique(np.asarray(m))
        assert set(u) <= {0, 255}, f"{r['id']}: mask not binary"
        assert (255 in u) == r.has_roi, f"{r['id']}: mask presence disagrees with has_roi"
print("cache checks passed: 384x640, binary masks, mask present iff coordinates exist")

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(20, 5))
for ax, (_, r) in zip(axes, mias_df[mias_df.has_roi].sample(8, random_state=1).iterrows()):
    a = np.asarray(Image.open(r[H.FULL_HIRES_COL]))
    rgb = np.stack([a] * 3, -1)
    rgb[outline(np.asarray(Image.open(r[H.ROI_HIRES_COL])))] = (57, 219, 92)
    ax.imshow(rgb)
    ax.set_title(f"{r['id']} {'M' if r.label else 'B'}", fontsize=9)
    ax.axis("off")
plt.suptitle("Cached 384x640 model inputs with lesion masks")
plt.tight_layout()
plt.show()

## 5. Load the deployed checkpoints

Same files and thresholds as `app/inference.py`: the four v4 control checkpoints, with DenseNet121
served from the high-sensitivity variant whose metadata carries `decision_threshold = 0.305`.

In [ ]:
CKPT_DIR = os.path.join(ROOT, "models", "v4_hires_control_gap")
EXP2_CKPT_DIR = os.path.join(ROOT, "notebooks", "experiment_2_optimization", "checkpoints")
PRIMARY_ARCH = "densenet121"
CKPT_PATHS = {a: os.path.join(CKPT_DIR, f"hires_control_gap_a0.5_{a}.pth") for a in H.ARCHITECTURES}
CKPT_PATHS[PRIMARY_ARCH] = os.path.join(
    EXP2_CKPT_DIR, "hires_control_gap_a0.5_densenet121_high_sensitivity_t0.305.pth")


def load_model(arch):
    ck = torch.load(CKPT_PATHS[arch], map_location=H.DEVICE, weights_only=False)
    assert list(ck.get("input_size", [])) == [H.HIRES_H, H.HIRES_W], ck.get("input_size")
    m = H.SingleInputModel(arch, pooling=ck.get("pooling", "gap"), use_seg=ck.get("use_seg", False))
    m.load_state_dict(ck["model_state_dict"])
    return m.to(H.DEVICE).eval(), float(ck.get("decision_threshold") or 0.5)


THRESHOLDS = {}
for arch in H.ARCHITECTURES:
    _, THRESHOLDS[arch] = load_model(arch)
    H.free_gpu()
print("deployed thresholds:", THRESHOLDS)

In [ ]:
@torch.no_grad()
def predict(model, df, batch_size=H.BATCH_SIZE):
    dl = DataLoader(H.HiResDataset(df, train=False, seg_out_hw=None), batch_size=batch_size,
                    shuffle=False, num_workers=H.NUM_WORKERS)
    probs = []
    for full, _, _, _ in tqdm(dl, leave=False):
        probs.append(torch.softmax(model(full.to(H.DEVICE)).float(), 1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

## 6. Plumbing check on CBIS-DDSM

Before trusting any MIAS number, the same `load_model` + `predict` path must reproduce the recorded
CBIS-DDSM test AUC. If it does, any drop on MIAS is the dataset, not a bug in this notebook.

In [ ]:
cbis_test = pd.read_csv(os.path.join(H.DF_DIR, "test_df_hires.csv"))
runs = pd.read_csv(os.path.join(ROOT, "notebooks", "experiment_1_baseline", "results", "hires_runs.csv"))
recorded = runs.set_index("architecture")

model, _ = load_model(PRIMARY_ARCH)
p = predict(model, cbis_test)
del model; H.free_gpu()
y = cbis_test[H.LABEL_COL].map(H.encode_label).values
auc = H.compute_metrics(y, (p >= 0.5).astype(int), p)["roc_auc"]
ref = recorded.loc[PRIMARY_ARCH, "roc_auc"]
print(f"DenseNet121 CBIS-DDSM test AUC: reproduced {auc:.4f} vs recorded {ref:.4f}")
assert abs(auc - ref) < 0.005, "inference path does not reproduce the recorded result"

## 7. Classification on MIAS

In [ ]:
y_mias = mias_df["label"].values
probs = {}
for arch in H.ARCHITECTURES:
    path = os.path.join(RESULTS_DIR, f"mias_probs_{arch}.npz")
    model, _ = load_model(arch)
    probs[arch] = predict(model, mias_df)
    del model; H.free_gpu()
    np.savez_compressed(path, ids=mias_df["id"].values, labels=y_mias, probs=probs[arch])
print("saved per-image probabilities to", RESULTS_DIR)

In [ ]:
def cluster_bootstrap_ci(labels, probs, clusters, threshold, metrics, n_boot=2000, seed=H.SEED):
    """95% CI resampling WOMEN (clusters) with replacement, not images. Modelled on
    hires_lib.bootstrap_ci; one pass computes every metric from the same resamples."""
    rng = np.random.default_rng(seed)
    labels, probs, clusters = map(np.asarray, (labels, probs, clusters))
    uniq = np.unique(clusters)
    idx_of = {c: np.flatnonzero(clusters == c) for c in uniq}
    vals = {m: [] for m in metrics}
    for _ in range(n_boot):
        i = np.concatenate([idx_of[c] for c in rng.choice(uniq, len(uniq))])
        if len(np.unique(labels[i])) < 2:
            continue
        r = H.metrics_at_threshold(labels[i], probs[i], threshold)
        for m in metrics:
            vals[m].append(r[m])
    return {m: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for m, v in vals.items()}


METRICS = ["accuracy", "roc_auc", "sensitivity", "specificity", "f1"]
rows = []
for arch in H.ARCHITECTURES:
    for kind, t in (("deployed", THRESHOLDS[arch]), ("t=0.50", 0.5)):
        if kind == "t=0.50" and t == THRESHOLDS[arch]:
            continue
        pt = H.metrics_at_threshold(y_mias, probs[arch], t)
        ci = cluster_bootstrap_ci(y_mias, probs[arch], mias_df["woman_id"], t, METRICS)
        row = {"model": H.ARCH_DISPLAY_NAMES[arch], "arch": arch, "threshold_kind": kind,
               "threshold": t, "n": len(y_mias)}
        for m in METRICS:
            row[m], row[f"{m}_lo"], row[f"{m}_hi"] = pt[m], *ci[m]
        rows.append(row)
mias_metrics = pd.DataFrame(rows)
mias_metrics.to_csv(os.path.join(RESULTS_DIR, "mias_classification_metrics.csv"), index=False)


def fmt(r, m):
    return f"{100 * r[m]:.1f} [{100 * r[m + '_lo']:.1f}-{100 * r[m + '_hi']:.1f}]"


pd.DataFrame({"model": mias_metrics["model"], "threshold": mias_metrics["threshold"].round(3),
              **{m: [fmt(r, m) for _, r in mias_metrics.iterrows()] for m in METRICS}})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for arch in H.ARCHITECTURES:
    fpr, tpr, _ = roc_curve(y_mias, probs[arch])
    r = mias_metrics[(mias_metrics.arch == arch)].iloc[0]
    ax.plot(fpr, tpr, lw=2, label=f"{H.ARCH_DISPLAY_NAMES[arch]} (AUC {r.roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
ax.set(xlabel="1 - specificity", ylabel="sensitivity", title=f"ROC on MIAS (n={len(y_mias)})")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "mias_roc.png"), dpi=200)
plt.show()

In [ ]:
# Which abnormality types transfer? Accuracy at the deployed threshold, per MIAS class. Counts are
# small -- read this as descriptive, not as evidence of a difference.
by_class = []
for arch in H.ARCHITECTURES:
    pred = (probs[arch] >= THRESHOLDS[arch]).astype(int)
    for cls, g in mias_df.assign(correct=(pred == y_mias)).groupby("abnorm_class"):
        by_class.append({"model": H.ARCH_DISPLAY_NAMES[arch], "abnorm_class": cls, "n": len(g),
                         "n_malignant": int(g.label.sum()), "accuracy": g.correct.mean()})
by_class = pd.DataFrame(by_class)
by_class.to_csv(os.path.join(RESULTS_DIR, "mias_accuracy_by_class.csv"), index=False)
by_class.pivot_table(index=["abnorm_class", "n", "n_malignant"], columns="model",
                     values="accuracy").round(3)

## 8. Internal (CBIS-DDSM) vs external (MIAS)

Both columns at threshold 0.50 so every architecture is compared on the same footing (the recorded
CBIS-DDSM numbers in `hires_runs.csv` are at 0.50). AUC is threshold-free.

In [ ]:
cmp_rows = []
for arch in H.ARCHITECTURES:
    ext = H.metrics_at_threshold(y_mias, probs[arch], 0.5)
    for m in ["accuracy", "roc_auc", "sensitivity", "specificity"]:
        cmp_rows.append({"model": H.ARCH_DISPLAY_NAMES[arch], "metric": m,
                         "cbis_ddsm_test": recorded.loc[arch, m], "mias": ext[m],
                         "delta": ext[m] - recorded.loc[arch, m]})
cmp = pd.DataFrame(cmp_rows)
cmp.to_csv(os.path.join(RESULTS_DIR, "external_vs_internal.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, m in zip(axes, ["roc_auc", "accuracy"]):
    sub = cmp[cmp.metric == m]
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub.cbis_ddsm_test, 0.4, label="CBIS-DDSM test (internal)")
    ax.bar(x + 0.2, sub.mias, 0.4, label="MIAS (external)")
    for xi, (a, b) in enumerate(zip(sub.cbis_ddsm_test, sub.mias)):
        ax.text(xi - 0.2, a + 0.01, f"{a:.3f}", ha="center", fontsize=8)
        ax.text(xi + 0.2, b + 0.01, f"{b:.3f}", ha="center", fontsize=8)
    ax.set_xticks(x, sub.model)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--")
    ax.set_title({"roc_auc": "ROC-AUC", "accuracy": "Accuracy (t = 0.50)"}[m])
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "external_vs_internal.png"), dpi=200)
plt.show()
cmp.pivot(index="model", columns="metric", values="delta").round(3)

## 9. Grad-CAM localization on MIAS

Same method as `explainability/gradcam_validation.ipynb`: Grad-CAM for the malignant class on the
extractor output, AMP off, scored inside the non-padding region against the lesion mask.

**Caveat when comparing with CBIS-DDSM:** MIAS ROIs are enclosing *circles*, typically far larger
than the lesion outline, so the chance rate is higher and hit rates are easier to achieve. The
`conc_ratio_x` column (energy concentration divided by chance) corrects for ROI size and is the
fairest single number to compare across the two datasets.

In [ ]:
loc_df = mias_df[mias_df.has_roi].reset_index(drop=True)
print(f"localization subset: {len(loc_df)} images")

per_image = []
for arch in H.ARCHITECTURES:
    model, _ = load_model(arch)
    with H.GradCAM(model) as cam_fn:
        for _, r in tqdm(loc_df.iterrows(), total=len(loc_df), desc=arch, leave=False):
            x = H._to_rgb_tensor(Image.open(r[H.FULL_HIRES_COL]).convert("L"))[None].to(H.DEVICE)
            cam, pr = cam_fn(x, class_idx=1)
            mask = np.asarray(Image.open(r[H.ROI_HIRES_COL])) > 127
            valid = H.validity_mask(r.orig_w, r.orig_h)
            m = H.localization_metrics(cam[0], mask, valid)
            m["iou@0.5"] = H.iou_at(cam[0], mask, valid, 0.5)
            per_image.append({"arch": arch, "id": r["id"], "label": r.label,
                              "abnorm_class": r.abnorm_class, "p_malignant": float(pr[0, 1]), **m})
    del model; H.free_gpu()

per_image = pd.DataFrame(per_image)
per_image.to_csv(os.path.join(RESULTS_DIR, "mias_localization_per_image.csv"), index=False)


def summarise(g):
    return pd.Series({
        "n": len(g),
        "pointing_%": 100 * g.pointing_hit.mean(),
        "pointing@15px_%": 100 * g.pointing_hit_tol.mean(),
        "energy_conc": g.energy_concentration.mean(),
        "conc_ratio_x": g.concentration_ratio.mean(),
        "pixel_ap": g.pixel_ap.mean(),
        "pixel_auroc": g.pixel_auroc.mean(),
        "iou@0.5": g["iou@0.5"].mean(),
        "peak_in_padding_%": 100 * g.peak_in_padding.mean(),
        "median_peak_dist_px": g.peak_dist_px.median(),
        "chance_rate": g.chance_rate.mean(),
    })


loc_mias = per_image.groupby("arch").apply(summarise, include_groups=False).reset_index().rename(columns={"arch": "model"})
loc_mias.to_csv(os.path.join(RESULTS_DIR, "mias_localization_summary.csv"), index=False)
loc_mias.round(3)

In [ ]:
loc_cbis = pd.read_csv(os.path.join(ROOT, "notebooks", "explainability", "results",
                                    "localization_summary_single_all_archs.csv"))
cols = ["pointing_%", "energy_conc", "conc_ratio_x", "pixel_auroc"]
loc_cmp = (loc_cbis.set_index("model")[cols].add_suffix(" | CBIS")
           .join(loc_mias.set_index("model")[cols].add_suffix(" | MIAS")))
loc_cmp = loc_cmp[[f"{c} | {d}" for c in cols for d in ("CBIS", "MIAS")]]
loc_cmp.index = loc_cmp.index.map(H.ARCH_DISPLAY_NAMES)
loc_cmp.to_csv(os.path.join(RESULTS_DIR, "localization_external_vs_internal.csv"))
print("mean chance rate (ROI share of breast) on MIAS:",
      f"{loc_mias.chance_rate.mean():.3f}")
loc_cmp.round(3)

In [ ]:
# Best and worst MIAS examples for the primary (deployed) model, by energy concentration.
sub = per_image[per_image.arch == PRIMARY_ARCH].sort_values("energy_concentration", ascending=False)
picks = pd.concat([sub.head(4), sub.tail(4)])
model, _ = load_model(PRIMARY_ARCH)
fig, axes = plt.subplots(2, 4, figsize=(14, 10))
with H.GradCAM(model) as cam_fn:
    for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
        row = loc_df[loc_df["id"] == r["id"]].iloc[0]
        a = np.asarray(Image.open(row[H.FULL_HIRES_COL]).convert("L"))
        x = H._to_rgb_tensor(Image.fromarray(a))[None].to(H.DEVICE)
        cam, _ = cam_fn(x, class_idx=1)
        ax.imshow(a, cmap="gray")
        ax.imshow(np.ma.masked_where(~H.validity_mask(row.orig_w, row.orig_h), cam[0]),
                  cmap="jet", alpha=0.4)
        ax.contour(np.asarray(Image.open(row[H.ROI_HIRES_COL])) > 127, levels=[0.5],
                   colors="lime", linewidths=1.2)
        ax.set_title(f"{r['id']} {r.abnorm_class} {'M' if r.label else 'B'}\n"
                     f"p(M)={r.p_malignant:.2f}  EC={r.energy_concentration:.2f}", fontsize=9)
        ax.axis("off")
del model; H.free_gpu()
axes[0, 0].set_ylabel("best")
plt.suptitle(f"{H.ARCH_DISPLAY_NAMES[PRIMARY_ARCH]} Grad-CAM on MIAS -- top row best, "
             "bottom row worst (green = MIAS lesion circle)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "mias_gradcam_examples.png"), dpi=150)
plt.show()

## 10. Summary

In [ ]:
dep = mias_metrics[mias_metrics.threshold_kind == "deployed"].set_index("arch")
summary = pd.DataFrame({
    "model": [H.ARCH_DISPLAY_NAMES[a] for a in H.ARCHITECTURES],
    "threshold": [THRESHOLDS[a] for a in H.ARCHITECTURES],
    "MIAS AUC [95% CI]": [fmt(dep.loc[a], "roc_auc") for a in H.ARCHITECTURES],
    "MIAS accuracy": [fmt(dep.loc[a], "accuracy") for a in H.ARCHITECTURES],
    "MIAS sensitivity": [fmt(dep.loc[a], "sensitivity") for a in H.ARCHITECTURES],
    "MIAS specificity": [fmt(dep.loc[a], "specificity") for a in H.ARCHITECTURES],
    "CBIS-DDSM AUC": [round(recorded.loc[a, "roc_auc"], 4) for a in H.ARCHITECTURES],
    "MIAS pointing %": [round(loc_mias.set_index("model").loc[a, "pointing_%"], 1)
                        for a in H.ARCHITECTURES],
    "MIAS conc. ratio": [round(loc_mias.set_index("model").loc[a, "conc_ratio_x"], 2)
                         for a in H.ARCHITECTURES],
})
summary.to_csv(os.path.join(RESULTS_DIR, "mias_summary.csv"), index=False)
summary